In [7]:
!pip install pdfplumber

# TAHAP 1: MEMBANGUN CASE BASE
**Deskripsi:** Melakukan akuisisi data (PDF), ekstraksi teks, pembersihan data (cleaning), dan validasi keutuhan teks.

In [34]:
import os
from google.colab import files

DATA_DIRS = ['data/raw/', 'data/cleaned/', 'input_pdfs']

def setup_environment(dirs):
    for d in dirs:
        if not os.path.exists(d):
            os.makedirs(d)
            print(f"Direktori created: {d}")
        else:
            print(f"Direktori sudah ada: {d}")

setup_environment(DATA_DIRS)

DATA_RAW = 'data/raw/'
DATA_CLEANED = 'data/cleaned/'
PDF_FOLDER = 'input_pdfs'

Direktori sudah ada: data/raw/
Direktori sudah ada: data/cleaned/
Direktori sudah ada: input_pdfs


In [35]:
import os
import re
import pdfplumber
import shutil
from google.colab import files

def clear_folder(folder_path):
    if os.path.exists(folder_path):
        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)
            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.unlink(file_path)
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
            except Exception as e:
                print(f'Gagal menghapus {file_path}: {e}')

def upload_pdf_files(target_folder):
    if not os.path.exists(target_folder):
        os.makedirs(target_folder)
    uploaded = files.upload()
    for filename, content in uploaded.items():
        path = os.path.join(target_folder, filename)
        with open(path, 'wb') as f:
            f.write(content)
        print(f"[OK] {filename}")

def clean_legal_text(text):
    text = text.lower()
    patterns = [
        r'halaman \d+ dari \d+',
        r'direktori putusan mahkamah agung republik indonesia',
        r'putusan\.mahkamahagung\.go\.id',
        r'disclamer : .*$',
        r'kepaniteraan mahkamah agung republik indonesia',
        r'mahkamah agung republik indonesia',
    ]
    for pattern in patterns:
        text = re.sub(pattern, '', text, flags=re.MULTILINE | re.IGNORECASE)
    text = re.sub(r'[^a-z0-9\s/.,]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def process_cases(source_folder, cleaned_folder, raw_folder, clear_first=True):
    if clear_first:
        clear_folder(cleaned_folder)
        clear_folder(raw_folder)

    if not os.path.exists(cleaned_folder): os.makedirs(cleaned_folder)
    if not os.path.exists(raw_folder): os.makedirs(raw_folder)

    pdf_files = [f for f in os.listdir(source_folder) if f.endswith('.pdf')]

    for i, filename in enumerate(pdf_files, start=1):
        new_name_txt = f"case_{i:03d}.txt"
        new_name_pdf = f"case_{i:03d}.pdf"
        src_path = os.path.join(source_folder, filename)
        shutil.copy(src_path, os.path.join(raw_folder, new_name_pdf))

        try:
            all_text = ""
            with pdfplumber.open(src_path) as pdf:
                for page in pdf.pages:
                    page_text = page.extract_text()
                    if page_text: all_text += page_text + "\n"

            cleaned_content = clean_legal_text(all_text)
            with open(os.path.join(cleaned_folder, new_name_txt), 'w', encoding='utf-8') as f:
                f.write(cleaned_content)
            print(f"[PROSES] {filename} -> {new_name_txt}")
        except Exception as e:
            print(f"[ERROR] {filename}: {str(e)}")

In [37]:
import os
import requests

def download_all_from_github(target_folder):
    if not os.path.exists(target_folder):
        os.makedirs(target_folder)

    api_url = "https://api.github.com/repos/didan-ui/Case-Based-Reasoning/contents/data/raw"
    raw_base_url = "https://raw.githubusercontent.com/didan-ui/Case-Based-Reasoning/main/data/raw/"

    response = requests.get(api_url)
    if response.status_code == 200:
        contents = response.json()
        pdf_files = [file['name'] for file in contents if file['name'].endswith('.pdf')]
        for filename in pdf_files:
            file_res = requests.get(raw_base_url + filename)
            if file_res.status_code == 200:
                with open(os.path.join(target_folder, filename), 'wb') as f:
                    f.write(file_res.content)

clear_folder(PDF_FOLDER)
download_all_from_github(PDF_FOLDER)
clear_folder(DATA_RAW)
clear_folder(DATA_CLEANED)
process_cases(PDF_FOLDER, DATA_CLEANED, DATA_RAW, clear_first=False)

for f in os.listdir('/content/'):
    if f.endswith('.pdf') and ('(' in f or 'putusan' in f.lower()):
        try: os.remove(os.path.join('/content/', f))
        except: pass

[PROSES] putusan_336_pid.sus_2025_pn_yyk_20260626133341.pdf -> case_001.txt
[PROSES] putusan_418_pid.sus_2025_pn_yyk_20260626133229.pdf -> case_002.txt
[PROSES] putusan_303_pid.sus_2025_pn_yyk_20260626133419.pdf -> case_003.txt
[PROSES] putusan_381_pid.sus_2025_pn_yyk_20260626133355.pdf -> case_004.txt
[PROSES] putusan_346_pid.sus_2025_pn_yyk_20260626133317.pdf -> case_005.txt
[PROSES] putusan_353_pid.sus_2025_pn_yyk_20260626133329.pdf -> case_006.txt
[PROSES] putusan_367_pid.sus_2025_pn_yyk_20260626133256.pdf -> case_007.txt
[PROSES] putusan_324_pid.sus_2025_pn_yyk_20260626133630.pdf -> case_008.txt
[PROSES] putusan_366_pid.sus_2025_pn_yyk_20260626133248.pdf -> case_009.txt
[PROSES] putusan_396_pid.sus_2025_pn_yyk_20260626133233.pdf -> case_010.txt
[PROSES] putusan_340_pid.sus_2025_pn_yyk_20260626133601.pdf -> case_011.txt
[PROSES] putusan_304_pid.sus_2025_pn_yyk_20260626133410.pdf -> case_012.txt
[PROSES] putusan_397_pid.sus_2025_pn_yyk_20260626133237.pdf -> case_013.txt
[PROSES] put

In [11]:
import os

def validate_cleaning(cleaned_folder, log_path='cleaning.log'):
    if not os.path.exists('logs'): os.makedirs('logs')
    log_file = os.path.join('logs', log_path)

    print(f"--- Memulai Validasi di {cleaned_folder} ---")
    with open(log_file, 'w') as log:
        for filename in sorted(os.listdir(cleaned_folder)):
            if filename.endswith('.txt'):
                path = os.path.join(cleaned_folder, filename)
                with open(path, 'r', encoding='utf-8') as f:
                    content = f.read()

                # Kriteria 1: Periksa keutuhan teks (contoh: minimal 1000 karakter atau cek keyword)
                char_count = len(content)
                has_essential = 'menimbang' in content and 'mengadili' in content
                status = "VALID" if (char_count > 1000 and has_essential) else "PERLU CEK MANUAL"

                log_msg = f"[{status}] {filename}: {char_count} karakter, Keywords found: {has_essential}\n"
                log.write(log_msg)
                print(log_msg.strip())

# Jalankan validasi setelah proses sinkronisasi selesai
validate_cleaning(DATA_CLEANED)

--- Memulai Validasi di data/cleaned/ ---
[VALID] case_001.txt: 145258 karakter, Keywords found: True
[VALID] case_002.txt: 52173 karakter, Keywords found: True
[VALID] case_003.txt: 109990 karakter, Keywords found: True
[VALID] case_004.txt: 57071 karakter, Keywords found: True
[VALID] case_005.txt: 119993 karakter, Keywords found: True
[VALID] case_006.txt: 96634 karakter, Keywords found: True
[VALID] case_007.txt: 93383 karakter, Keywords found: True
[VALID] case_008.txt: 54600 karakter, Keywords found: True
[VALID] case_009.txt: 101918 karakter, Keywords found: True
[VALID] case_010.txt: 205746 karakter, Keywords found: True
[VALID] case_011.txt: 67648 karakter, Keywords found: True
[VALID] case_012.txt: 104970 karakter, Keywords found: True
[VALID] case_013.txt: 115819 karakter, Keywords found: True
[VALID] case_014.txt: 87225 karakter, Keywords found: True
[VALID] case_015.txt: 74799 karakter, Keywords found: True


In [12]:
# Menampilkan isi cleaning log untuk verifikasi output Tahap 1
log_path = 'logs/cleaning.log'

if os.path.exists(log_path):
    print(f"--- Isi File: {log_path} ---\n")
    with open(log_path, 'r') as f:
        print(f.read())
else:
    print("File log belum ditemukan. Pastikan sel validasi sudah dijalankan.")

--- Isi File: logs/cleaning.log ---

[VALID] case_001.txt: 145258 karakter, Keywords found: True
[VALID] case_002.txt: 52173 karakter, Keywords found: True
[VALID] case_003.txt: 109990 karakter, Keywords found: True
[VALID] case_004.txt: 57071 karakter, Keywords found: True
[VALID] case_005.txt: 119993 karakter, Keywords found: True
[VALID] case_006.txt: 96634 karakter, Keywords found: True
[VALID] case_007.txt: 93383 karakter, Keywords found: True
[VALID] case_008.txt: 54600 karakter, Keywords found: True
[VALID] case_009.txt: 101918 karakter, Keywords found: True
[VALID] case_010.txt: 205746 karakter, Keywords found: True
[VALID] case_011.txt: 67648 karakter, Keywords found: True
[VALID] case_012.txt: 104970 karakter, Keywords found: True
[VALID] case_013.txt: 115819 karakter, Keywords found: True
[VALID] case_014.txt: 87225 karakter, Keywords found: True
[VALID] case_015.txt: 74799 karakter, Keywords found: True



In [13]:
import os

def check_text_integrity(cleaned_folder):
    print(f"=== LAPORAN KEUTUHAN TEKS (OUTPUT TAHAP 1) ===\n")

    for filename in sorted(os.listdir(cleaned_folder)):
        if filename.endswith('.txt'):
            path = os.path.join(cleaned_folder, filename)
            with open(path, 'r', encoding='utf-8') as f:
                content = f.read()

            char_count = len(content)
            # Estimasi sederhana: dokumen hukum rata-rata > 5000 karakter
            # Kita hitung persentase terhadap ambang batas kelayakan
            integrity_score = min(100, (char_count / 5000) * 100)

            has_menimbang = 'menimbang' in content.lower()
            has_mengadili = 'mengadili' in content.lower()

            print(f"File: {filename}")
            print(f"- Estimasi Keutuhan: {integrity_score:.2f}%")
            print(f"- Struktur Menimbang: {'[OK]' if has_menimbang else '[MISSING]'}")
            print(f"- Struktur Mengadili: {'[OK]' if has_mengadili else '[MISSING]'}")
            print(f"- Status Akhir: {'MEMENUHI SYARAT (>= 80%)' if integrity_score >= 80 and has_menimbang and has_mengadili else 'PERLU REVISI'}")
            print("-" * 40)

check_text_integrity(DATA_CLEANED)

=== LAPORAN KEUTUHAN TEKS (OUTPUT TAHAP 1) ===

File: case_001.txt
- Estimasi Keutuhan: 100.00%
- Struktur Menimbang: [OK]
- Struktur Mengadili: [OK]
- Status Akhir: MEMENUHI SYARAT (>= 80%)
----------------------------------------
File: case_002.txt
- Estimasi Keutuhan: 100.00%
- Struktur Menimbang: [OK]
- Struktur Mengadili: [OK]
- Status Akhir: MEMENUHI SYARAT (>= 80%)
----------------------------------------
File: case_003.txt
- Estimasi Keutuhan: 100.00%
- Struktur Menimbang: [OK]
- Struktur Mengadili: [OK]
- Status Akhir: MEMENUHI SYARAT (>= 80%)
----------------------------------------
File: case_004.txt
- Estimasi Keutuhan: 100.00%
- Struktur Menimbang: [OK]
- Struktur Mengadili: [OK]
- Status Akhir: MEMENUHI SYARAT (>= 80%)
----------------------------------------
File: case_005.txt
- Estimasi Keutuhan: 100.00%
- Struktur Menimbang: [OK]
- Struktur Mengadili: [OK]
- Status Akhir: MEMENUHI SYARAT (>= 80%)
----------------------------------------
File: case_006.txt
- Estimasi Ke

In [31]:
import pdfplumber

# Memilih salah satu file untuk verifikasi
sample_pdf = os.path.join(PDF_FOLDER, os.listdir(PDF_FOLDER)[0])
sample_txt = os.path.join(DATA_CLEANED, 'case_001.txt')

print(f"--- Verifikasi File: {sample_pdf} ---\n")

# 1. Ambil sedikit teks mentah langsung dari PDF
with pdfplumber.open(sample_pdf) as pdf:
    raw_sample = pdf.pages[0].extract_text()[:500]

# 2. Ambil teks yang sudah dibersihkan
with open(sample_txt, 'r') as f:
    cleaned_sample = f.read()[:500]

print("=== CONTOH TEKS MENTAH (RAW) ===")
print(raw_sample)
print("\n=== CONTOH TEKS BERSIH (CLEANED) ===")
print(cleaned_sample)

print("\nAnalisis: Apakah struktur 'Putusan Nomor...' dan identitas pihak masih terbaca dengan jelas di teks bersih?")

--- Verifikasi File: input_pdfs/putusan_336_pid.sus_2025_pn_yyk_20260626133341.pdf ---

=== CONTOH TEKS MENTAH (RAW) ===
a
i
s
e
n
o
d
n
I
k
i
l
b
u
p
e
a
R
i
s
g e
n
n
o
u
d
g
n
A
I
h k
a i
l
m b
u
a
p
k
Direktori Putusan Mahkamah Agung Republik Indonesia
e
h a
putusan.mahkamahagung.go.id
R
a i
P U T U S A N s
M Nomor 336/Pid.Sus/2025/PN Yyk
g e
DEMIn KEADILAN BERDASARKAN KETUHANAN YANG MAHA ESA n
Pengadilan Negeri Yogyakarta yang mengadili perkara pidana doengan
u
acara pemeriksaan biasa dalam tingkat pertama menjatuhkan putusan sebagai
d
g
berikut dalam perkara Para Terdakwa:
n
ATerdakwa 1
1. Nama lengkap : Hid

=== CONTOH TEKS BERSIH (CLEANED) ===
a i s e n o d n i k i l b u p e a r i s g e n n o u d g n a i h k a i l m b u a p k e h a r a i p u t u s a n s m nomor 336/pid.sus/2025/pn yyk g e demin keadilan berdasarkan ketuhanan yang maha esa n pengadilan negeri yogyakarta yang mengadili perkara pidana doengan u acara pemeriksaan biasa dalam tingkat pertama menjatuhkan putusan seba

In [32]:
import pdfplumber
import os

# Memilih sampel kasus pertama
sample_pdf_path = os.path.join(DATA_RAW, 'case_001.pdf')
sample_txt_path = os.path.join(DATA_CLEANED, 'case_001.txt')

print(f"--- Verifikasi Perbandingan Kasus 001 ---\n")

# 1. Ekstraksi teks mentah dari halaman pertama PDF
with pdfplumber.open(sample_pdf_path) as pdf:
    raw_text_sample = pdf.pages[0].extract_text()[:600]

# 2. Membaca teks yang sudah dibersihkan
with open(sample_txt_path, 'r', encoding='utf-8') as f:
    cleaned_text_sample = f.read()[:600]

print("=== TEKS MENTAH (DARI PDF) ===")
print(raw_text_sample)
print("\n" + "="*30 + "\n")
print("=== TEKS BERSIH (DATA/CLEANED) ===")
print(cleaned_text_sample)

print("\n[Analisis] Periksa apakah nomor putusan dan identitas terdakwa masih ada dalam teks bersih.")

--- Verifikasi Perbandingan Kasus 001 ---

=== TEKS MENTAH (DARI PDF) ===
a
i
s
e
n
o
d
n
I
k
i
l
b
u
p
e
a
R
i
s
g e
n
n
o
u
d
g
n
A
I
h k
a i
l
m b
u
a
p
k
Direktori Putusan Mahkamah Agung Republik Indonesia
e
h a
putusan.mahkamahagung.go.id
R
a i
P U T U S A N s
M Nomor 336/Pid.Sus/2025/PN Yyk
g e
DEMIn KEADILAN BERDASARKAN KETUHANAN YANG MAHA ESA n
Pengadilan Negeri Yogyakarta yang mengadili perkara pidana doengan
u
acara pemeriksaan biasa dalam tingkat pertama menjatuhkan putusan sebagai
d
g
berikut dalam perkara Para Terdakwa:
n
ATerdakwa 1
1. Nama lengkap : Hidayat Dwi Sulistiyo Bin SupriyaIni (alm)
h2. Tempat lahir : Magelang k
a 3. Umur/Tanggal lahir : 24 t


=== TEKS BERSIH (DATA/CLEANED) ===
a i s e n o d n i k i l b u p e a r i s g e n n o u d g n a i h k a i l m b u a p k e h a r a i p u t u s a n s m nomor 336/pid.sus/2025/pn yyk g e demin keadilan berdasarkan ketuhanan yang maha esa n pengadilan negeri yogyakarta yang mengadili perkara pidana doengan u acara pemeriksaan 

# TAHAP 2: CASE REPRESENTATION
**Deskripsi:** Mentransformasi teks yang sudah dibersihkan ke dalam format terstruktur (JSON/CSV) dengan mengekstrak fitur kunci seperti nomor putusan, fakta hukum, dan amar.

In [39]:
import json
import glob
import re
import os

def save_to_structured_base(source_folder, output_file):
    case_base = []
    files_found = glob.glob(os.path.join(source_folder, "case_*.txt"))
    for filepath in sorted(files_found):
        with open(filepath, 'r', encoding='utf-8') as f:
            text = f.read()
            case_representation = represent_case_v2(text)
            case_base.append(case_representation)
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(case_base, f, indent=4)

structured_file = os.path.join(DATA_CLEANED, 'case_base.json')
save_to_structured_base(DATA_CLEANED, structured_file)

In [41]:
import re
import json
import glob
import os

def represent_case_v2(raw_text):
    nomor_match = re.search(r"putusan nomor (\d+/[^/]+/[^/]+/[^/]+)", raw_text, re.I)
    nomor_putusan = nomor_match.group(1) if nomor_match else "Tidak Diketahui"

    tanggal_match = re.search(r"tanggal\s+(\d+\s+[a-z]+\s+\d{4})", raw_text, re.I)
    tanggal = tanggal_match.group(1) if tanggal_match else "Tidak Terdeteksi"

    pihak_match = re.search(r"nama lengkap\s+([^;\n]+)", raw_text, re.I)
    pihak = pihak_match.group(1).strip() if pihak_match else "Tidak Terdeteksi"

    pasal_match = re.search(r"pasal\s+(\d+[^,.]*)", raw_text, re.I)
    pasal = pasal_match.group(0).strip() if pasal_match else "Tidak Terdeteksi"

    fakta_start = raw_text.find("menimbang")
    amar_start = raw_text.rfind("mengadili")

    if fakta_start != -1 and amar_start != -1 and amar_start > fakta_start:
        fakta_hukum = raw_text[fakta_start:amar_start].strip()
        amar_putusan = raw_text[amar_start:].strip()
    else:
        fakta_hukum = raw_text[fakta_start:fakta_start+5000] if fakta_start != -1 else raw_text[:5000]
        amar_putusan = raw_text[amar_start:] if amar_start != -1 else ""

    return {
        "nomor_putusan": nomor_putusan,
        "tanggal": tanggal,
        "pihak": pihak,
        "pasal": pasal,
        "fakta_hukum": fakta_hukum,
        "amar_putusan": amar_putusan,
        "kategori": "Pidana"
    }

In [42]:
import pandas as pd
import os

PROCESSED_DIR = 'data/processed/'
if not os.path.exists(PROCESSED_DIR):
    os.makedirs(PROCESSED_DIR)

def export_structured_data(case_base):
    json_path = os.path.join(PROCESSED_DIR, 'cases.json')
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(case_base, f, indent=4)

    csv_path = os.path.join(PROCESSED_DIR, 'cases.csv')
    df = pd.DataFrame(case_base)
    cols = ['nomor_putusan', 'tanggal', 'pihak', 'pasal', 'ringkasan_fakta', 'word_count', 'kategori']
    df_csv = df[[c for c in cols if c in df.columns]]
    df_csv.to_csv(csv_path, index=False, sep=';')

    print(f"Data saved to: {json_path} and {csv_path}")
    return df_csv

In [17]:
import glob
import os
import pandas as pd

# Memastikan fungsi represent_case_v3 didefinisikan dengan benar
def represent_case_v3(raw_text):
    # Mengambil data dasar dari versi sebelumnya
    base_data = represent_case_v2(raw_text)
    words = base_data['fakta_hukum'].split()
    word_count = len(words)
    ringkasan = base_data['fakta_hukum'][:200] + "..." if len(base_data['fakta_hukum']) > 200 else base_data['fakta_hukum']

    base_data.update({
        "word_count": word_count,
        "ringkasan_fakta": ringkasan
    })
    return base_data

def process_to_final_storage(source_folder):
    case_base = []
    files_found = glob.glob(os.path.join(source_folder, "case_*.txt"))
    for filepath in sorted(files_found):
        with open(filepath, 'r', encoding='utf-8') as f:
            text = f.read()
            case_rep = represent_case_v3(text)
            case_base.append(case_rep)

    # Memanggil fungsi export yang didefinisikan di sel f74fa13d
    return export_structured_data(case_base)

# Jalankan eksekusi
if 'DATA_CLEANED' in globals() and 'export_structured_data' in globals():
    df_final = process_to_final_storage(DATA_CLEANED)
    display(df_final.head())
else:
    print("Pastikan sel f74fa13d (export_structured_data) sudah dijalankan terlebih dahulu.")

[BERHASIL] Data diekspor ke:
- data/processed/cases.json
- data/processed/cases.csv


,nomor_putusan,tanggal,pihak,pasal,ringkasan_fakta,word_count,kategori
0,336/pid.sus/2025/pdn yyk n a i h k disclaimer ...,26 september 2025,hidayat dwi sulistiyo bin supriyaini alm h2. t...,pasal 127 ayat 1 huruf a undang undang a ri no...,"menimbang, bahw a para terdakwa diajukan ke pe...",24877,Pidana
1,418/pid.sus/2025/pdn yyk a n i h k disclaimer ...,20 oktober 2025,muhammad galang wibowo bin istanta h k 2. temp...,pasal 436 ayat 2 undang undang republik indone...,"menimbang, b ahwa terdakwa diajukan ke persida...",8570,Pidana
2,303/pid.sus/2025/pn yyk n a i h k disclaimer k...,3 agustus 2025,sonny williams jonathan dawir alias williams a...,pasal 132d ayat 1 g uu nomor 35 tahun 2009 ten...,"menimbang, bahwa terdakwa diajukan ke persidan...",18592,Pidana
3,381/pid.sus/2025/pn yyk n a i h k disclaimer k...,2 oktober 2025,mujais bin suharib 2. tempat lahir sumenep i h...,pasal 112 ayat n2 no,"menimbang, bahwa terdakwa diajukan ke persidan...",911,Pidana
4,346/pid.sus/2025/pn yyk d g n a i h k disclaim...,12 agustus 2025,arief faizal ahmad als. kriteng bin undirin bi...,pasal 1l45 ayat 1 yang terkait m b dengan sedi...,"menimbang, bahwa terdakwa diajukan ke persidan...",3326,Pidana


In [33]:
# Menghitung jumlah kasus per kategori
category_counts = df_final['kategori'].value_counts()

print("=== Jumlah Kasus Per Kategori ===")
display(category_counts)

=== Jumlah Kasus Per Kategori ===


,count
kategori,
Pidana,15


# TAHAP 3: CASE RETRIEVAL
**Deskripsi:** Mengimplementasikan pencarian kemiripan kasus menggunakan metode seperti TF-IDF dan Cosine Similarity untuk menemukan kasus terdahulu yang relevan.

In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import json
import os

id_stop_words = [
    'yang', 'untuk', 'pada', 'ke', 'para', 'namun', 'menurut', 'antara', 'dia',
    'dua', 'ia', 'seperti', 'jika', 'sehingga', 'kembali', 'dan', 'tidak', 'ini',
    'karena', 'kepada', 'oleh', 'saat', 'harus', 'sementara', 'setelah', 'belum',
    'kami', 'sekitar', 'bagi', 'serta', 'daripada', 'dengan', 'adalah', 'bahwa'
]

def load_case_base(filepath):
    if not os.path.exists(filepath):
        return []
    with open(filepath, 'r', encoding='utf-8') as f:
        return json.load(f)

def retrieve(query, k=5):
    cases = load_case_base(structured_file)
    if not cases:
        return "Case base kosong."

    corpus = [c['fakta_hukum'] if c['fakta_hukum'].strip() != "" else "kosong" for c in cases]

    try:
        vectorizer = TfidfVectorizer(
            stop_words=id_stop_words,
            token_pattern=r"(?u)\b\w\w+\b"
        )
        tfidf_matrix = vectorizer.fit_transform(corpus)
        query_vec = vectorizer.transform([query])
        similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()

        k_adj = min(k, len(cases))
        top_indices = similarities.argsort()[-k_adj:][::-1]

        results = []
        for idx in top_indices:
            results.append({
                "score": float(similarities[idx]),
                "case": cases[idx]
            })
        return results
    except ValueError as e:
        return f"Error retrieval: {str(e)}"

In [21]:
# Uji Coba Tahap 3: Retrieval (Sudah diperbaiki untuk menangani field opsional)
query_hukum = "Terdakwa melakukan tindak pidana penyalahgunaan narkotika golongan I bagi diri sendiri"

print(f"Mencari kasus serupa untuk query: '{query_hukum}'\n")
hasil_retrieval = retrieve(query_hukum, k=2)

if isinstance(hasil_retrieval, list):
    for i, res in enumerate(hasil_retrieval, 1):
        print(f"Hasil #{i} (Skor Kemiripan: {res['score']:.4f})")
        print(f"- Nomor Putusan: {res['case'].get('nomor_putusan', 'N/A')}")
        print(f"- Pasal: {res['case'].get('pasal', 'Tidak terdeteksi')}")
        print(f"- Cuplikan Fakta: {res['case'].get('fakta_hukum', '')[:200]}...")
        print("-" * 30)
else:
    print(hasil_retrieval)

Mencari kasus serupa untuk query: 'Terdakwa melakukan tindak pidana penyalahgunaan narkotika golongan I bagi diri sendiri'

Hasil #1 (Skor Kemiripan: 0.2018)
- Nomor Putusan: 324/pid.sus/2025/pdn yyk n a i h k disclaimer kepaniteraan mahkamah agunag republik indonesia berusaha untuk selalu mencantumkan informasi paling kini dan akurat sebagai bentuk komitmen mahkamah agung uintuk pelayanan publik, transparansi dan akuntabilitas pelaksanaan fungsi peradilan. namun dalam hal hal tertentu masih dimungkinkan terjadi permasalahan teknis terkait dengan akurasi dan keterkinian informasi yang klami sajikan, hal mana akan terus kami perbaiki dari waktu kewaktu. d em al a am il h k a e l p a a n n d it a e rm m aa e n n em m u a k h a k n a m ina a k h u a r g a u s n i g in .g fo o r . m id a s i t y e a lp n g 0 te 2 r 1 m 3 u 8 a 4 t p 3 a 3 d 4 a 8 s i e tu x s t. 3 in 1 i 8 a tau informasi yang seharusnya ada, namun belum tersedia, maka harap sege b ra hubungi kepaniteraan mahkamah agung ri

# TAHAP 4 & 5: CASE SOLUTION, REVISION, EVALUATION, & RETENTION
**Deskripsi:** Memberikan prediksi solusi (Reuse), melakukan validasi/revisi (Revise), mengevaluasi akurasi sistem (Evaluation), dan menyimpan kasus baru ke basis pengetahuan (Retention).

In [38]:
from collections import Counter
from sklearn.metrics import classification_report, accuracy_score

def predict_outcome(query, k=3):
    retrieved_results = retrieve(query, k=k)
    if isinstance(retrieved_results, str) or not retrieved_results:
        return {"prediction": "Tidak ditemukan", "confidence": 0}

    categories = [res['case'].get('kategori', 'Tidak Terkategori') for res in retrieved_results]
    vote_result = Counter(categories).most_common(1)
    return {
        "prediction": vote_result[0][0],
        "confidence": vote_result[0][1] / len(categories),
        "top_match_score": retrieved_results[0]['score']
    }

def run_evaluation():
    test_cases = [
        {"query": "penyalahgunaan narkotika golongan 1 bagi diri sendiri", "expected": "Pidana"},
        {"query": "permufakatan jahat tanpa hak memiliki narkotika", "expected": "Pidana"},
        {"query": "pencurian dengan pemberatan", "expected": "Pidana"}
    ]

    y_true = [tc['expected'] for tc in test_cases]
    y_pred = []

    for tc in test_cases:
        res = predict_outcome(tc['query'])
        y_pred.append(res['prediction'])
        print(f"Query: {tc['query']} -> Result: {res['prediction']}")

    print("\n--- Classification Report ---")
    print(classification_report(y_true, y_pred, zero_division=0))

run_evaluation()

Query: penyalahgunaan narkotika golongan 1 bagi diri sendiri -> Result: Tidak ditemukan
Query: permufakatan jahat tanpa hak memiliki narkotika -> Result: Tidak ditemukan
Query: pencurian dengan pemberatan -> Result: Tidak ditemukan

--- Classification Report ---
                 precision    recall  f1-score   support

         Pidana       0.00      0.00      0.00       3.0
Tidak ditemukan       0.00      0.00      0.00       0.0

       accuracy                           0.00       3.0
      macro avg       0.00      0.00      0.00       3.0
   weighted avg       0.00      0.00      0.00       3.0



### Tahap 4: Case Revision (Revisi Solusi)
Dalam tahap ini, jika skor kemiripan (similarity score) di bawah ambang batas (threshold), sistem harus meminta masukan pakar atau melakukan penyesuaian manual karena hasil retrieval dianggap kurang meyakinkan.

In [26]:
def revise_solution(retrieval_results, threshold=0.15):
    """
    Memeriksa apakah hasil retrieval layak digunakan atau perlu revisi pakar.
    """
    if not retrieval_results or retrieval_results[0]['score'] < threshold:
        print("--- PERINGATAN: Skor kemiripan rendah ---")
        return "Perlu Revisi Pakar (Kasus Baru/Unik)"

    return retrieval_results[0]['case']['kategori']

### Tahap 5: Case Retention (Penyimpanan Kasus Baru)
Setelah solusi dikonfirmasi (baik dari retrieval atau hasil revisi), kasus tersebut disimpan kembali ke dalam `case_base.json` agar sistem semakin pintar.

In [27]:
def retain_case(new_case_data, filepath=structured_file):
    """
    Menambahkan kasus baru yang telah divalidasi ke dalam database JSON.
    """
    current_base = load_case_base(filepath)
    current_base.append(new_case_data)

    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(current_base, f, indent=4)

    print(f"[RETAIN] Kasus baru berhasil disimpan. Total kasus sekarang: {len(current_base)}")

### Simulasi Siklus Lengkap (Retrieve -> Reuse -> Revise -> Retain)

In [40]:
def revise_solution(retrieval_results, threshold=0.15):
    if not retrieval_results or retrieval_results[0]['score'] < threshold:
        return "Perlu Revisi Pakar (Kasus Baru/Unik)"
    return retrieval_results[0]['case'].get('kategori', 'Pidana')

def retain_case(new_case_data, filepath=structured_file):
    current_base = load_case_base(filepath)
    current_base.append(new_case_data)
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(current_base, f, indent=4)

query_baru = "Pencurian kendaraan bermotor di area parkir mall pada malam hari"
hasil = retrieve(query_baru, k=1)
solusi_final = revise_solution(hasil)

if solusi_final == "Perlu Revisi Pakar (Kasus Baru/Unik)":
    data_simpan = {
        "nomor_putusan": "NEW/2025/SIMULASI",
        "tanggal": "2025-06-04",
        "pihak": "Simulasi Terdakwa",
        "pasal": "Pasal 363 KUHP",
        "fakta_hukum": query_baru,
        "kategori": "Pidana Umum"
    }
    retain_case(data_simpan)
    print("Solusi: Pidana Umum (Retained)")
else:
    print(f"Solusi: {solusi_final}")

Solusi: Pidana Umum (Retained)


### Dokumentasi Proyek (README.md)

In [30]:
readme_content = """
# Sistem Case-Based Reasoning (CBR) Putusan Mahkamah Agung

Proyek ini mengimplementasikan siklus lengkap Case-Based Reasoning (Retrieve, Reuse, Revise, Retain) untuk klasifikasi dan analisis hukum pada dokumen Putusan Mahkamah Agung.

## 1. Persyaratan (Requirements)
Pastikan Anda memiliki Python 3.x dan menginstal pustaka berikut:
```bash
pip install pdfplumber pandas scikit-learn numpy requests
```

## 2. Struktur Folder
Sistem secara otomatis akan mengelola struktur folder berikut:
- `data/raw/`: Menyimpan file PDF asli.
- `data/cleaned/`: Menyimpan hasil ekstraksi teks (.txt) dan `case_base.json`.
- `data/processed/`: Menyimpan data terstruktur dalam format CSV dan JSON.
- `input_pdfs/`: Folder sementara untuk proses unduhan/unggah.

## 3. Cara Menjalankan Pipeline
1. **Tahap 1 (Acquisition):** Jalankan sel inisialisasi untuk mengunduh data dari repositori GitHub atau unggah secara manual.
2. **Tahap 2 (Representation):** Jalankan fungsi ekstraksi untuk mengubah PDF menjadi teks terstruktur.
3. **Tahap 3 (Retrieval):** Masukkan query fakta hukum pada fungsi `retrieve()` untuk mencari kasus serupa.
4. **Tahap 4 & 5 (Reuse, Evaluation, Retention):** Gunakan fungsi `predict_outcome()` untuk klasifikasi otomatis dan `retain_case()` untuk menyimpan kasus baru ke database.

## 4. Contoh Penggunaan
```python
query = "Terdakwa menyalahgunakan narkotika jenis sabu"
hasil = retrieve(query, k=1)
print(hasil)
```

## 5. Kontributor
- Nama Anggota Kelompok : Faza Abdillah (202310370311154) & Adhidhan Obiansyah (202310370311163)
- Mata Kuliah: Penalaran Komputer - C
"""

with open('README.md', 'w', encoding='utf-8') as f:
    f.write(readme_content.strip())

print("File README.md berhasil dibuat di direktori root.")

File README.md berhasil dibuat di direktori root.
